# EXP_009c: The Lucier Resonance — "The Spectrum"

## Project Context: LATENT MORPHOGENESIS
**Phase:** 1 (Active Probing)  
**Isomorphism:** `Resonant Frequency` / `Singularity` / `Body without Organs`

---

## Theoretical Premise

EXP_009a and 009b established the *empirical* phenomenon: iterative re-injection converges to an architectural resonant state. This notebook provides the **mathematical explanation**.

**The Punchline:**  
Repeated application of a linear operator to a vector is **power iteration** — the standard algorithm for finding the dominant eigenvector. The "resonant frequency" revealed by the Lucier loop is literally the **dominant singular vector** of the weight matrix. The convergence rate depends on the **spectral gap** (ratio between the largest and second-largest singular values).

| Lucier (Acoustic) | Math (Linear Algebra) | LLM (Latent) |
| :--- | :--- | :--- |
| Resonant frequency | Dominant eigenvalue | Principal singular value of W_OV |
| Convergence speed | Spectral gap (σ₁/σ₂) | How "peaked" the head's function is |
| Room shape | Operator spectrum | Weight matrix geometry |
| Multiple resonances | Multiple eigenvalues | Multiple latent "modes" |

## Hypothesis
The empirically observed resonant states from EXP_009a/b should have **high cosine similarity** (>0.9) to the dominant right singular vector of the corresponding W_OV matrix. This would confirm that the Lucier loop is equivalent to power iteration and that the architecture's "voice" is mathematically predictable from the weights alone.

In [ ]:
# ============================================================
# STEP 1: CALIBRATION
# ============================================================
import torch
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from transformer_lens import HookedTransformer
from IPython.display import Markdown, display
from tqdm.notebook import tqdm
import os
import warnings
warnings.filterwarnings('ignore')

device = "cuda" if torch.cuda.is_available() else "cpu"
model = HookedTransformer.from_pretrained("gpt2-small", device=device)
print(f"Running on: {device}")

In [ ]:
# ============================================================
# STEP 2: LOAD EMPIRICAL RESULTS (from EXP_009b)
# ============================================================

data_dir = os.path.join("..", "_DATA", "EXP_009")

try:
    empirical_data = torch.load(
        os.path.join(data_dir, "009b_head_loop_results.pt"),
        map_location="cpu", weights_only=False
    )
    print(f"✓ Loaded empirical data: {len(empirical_data)} heads")
    HAS_EMPIRICAL = True
except FileNotFoundError:
    print("⚠ No empirical data found. Run EXP_009b first for comparison.")
    print("  This notebook will still compute the spectral analysis independently.")
    HAS_EMPIRICAL = False

In [ ]:
# ============================================================
# STEP 3: EIGENDECOMPOSITION — The Spectral Fingerprint
# ============================================================

def get_ov_matrix(model, layer, head):
    """Extract the combined OV matrix for a specific head."""
    W_V = model.W_V[layer, head]  # (d_model, d_head)
    W_O = model.W_O[layer, head]  # (d_head, d_model)
    return W_V @ W_O              # (d_model, d_model)

def get_top_tokens(model, resid_vector, k=5):
    """Decode a residual stream vector into top-k token predictions."""
    logits = resid_vector.to(device) @ model.W_U + model.b_U
    probs = torch.softmax(logits, dim=-1)
    top_probs, top_indices = torch.topk(probs, k)
    tokens = [model.tokenizer.decode([idx]) for idx in top_indices]
    return list(zip(tokens, top_probs.cpu().tolist()))


# Compute SVD for all heads
spectral_data = {}  # (layer, head) -> {singular_values, U, Vh, ...}

print("Computing SVD for all 144 heads...")
pbar = tqdm(total=model.cfg.n_layers * model.cfg.n_heads)

for layer in range(model.cfg.n_layers):
    for head in range(model.cfg.n_heads):
        W_OV = get_ov_matrix(model, layer, head).float().cpu()
        
        # SVD: W_OV = U @ diag(S) @ Vh
        U, S, Vh = torch.linalg.svd(W_OV, full_matrices=False)
        
        # The dominant right singular vector (what the Lucier loop converges to)
        dominant_direction = Vh[0]  # first row of Vh = first right singular vector
        
        # Spectral gap: ratio of top-2 singular values
        spectral_gap = (S[0] / S[1]).item() if S[1] > 0 else float('inf')
        
        # What token does the dominant eigenvector predict?
        top_tokens = get_top_tokens(model, dominant_direction)
        
        spectral_data[(layer, head)] = {
            "singular_values": S.numpy(),
            "dominant_vector": dominant_direction,
            "spectral_gap": spectral_gap,
            "top_tokens": top_tokens,
            "top_k_sv": S[:10].numpy(),  # top 10 singular values
        }
        pbar.update(1)

pbar.close()
print(f"✓ SVD complete for {len(spectral_data)} heads.")

---
## 4. Visualization

### 4a. Spectral Gap Grid
The spectral gap (σ₁/σ₂) determines convergence speed. High gap = fast convergence = strongly resonant room.

In [ ]:
# ============================================================
# VIS 4a: SPECTRAL GAP GRID
# ============================================================

gap_grid = np.zeros((model.cfg.n_layers, model.cfg.n_heads))
for (layer, head), data in spectral_data.items():
    gap_grid[layer, head] = min(data["spectral_gap"], 10.0)  # clip for vis

fig_gap = px.imshow(
    gap_grid,
    x=[f"H{h}" for h in range(model.cfg.n_heads)],
    y=[f"L{l}" for l in range(model.cfg.n_layers)],
    color_continuous_scale="Plasma",
    title="Spectral Gap (σ₁/σ₂): Predicted Resonance Strength",
    labels={"color": "σ₁/σ₂"},
    text_auto=".2f",
    aspect="auto",
)
fig_gap.update_layout(template="plotly_dark", height=600)
fig_gap.show()

### 4b. Eigenvalue Spectra — Selected Heads
Comparing the 'harmonic profile' of different heads. A head with one dominant eigenvalue is like a pure-tone room. A flat spectrum is like an anechoic chamber.

In [ ]:
# ============================================================
# VIS 4b: EIGENVALUE SPECTRA for Selected Heads
# ============================================================

# Select heads spanning the spectral gap range
sorted_by_gap = sorted(spectral_data.keys(), key=lambda k: spectral_data[k]["spectral_gap"], reverse=True)
selected_heads = [
    sorted_by_gap[0],      # Highest gap (most resonant)
    sorted_by_gap[-1],     # Lowest gap (least resonant)
    (9, 9),                # The EXP_002 degenerate head
    (0, 0),                # First head
    sorted_by_gap[len(sorted_by_gap)//2],  # Median
]
seen = set()
selected_heads = [h for h in selected_heads if not (h in seen or seen.add(h))]

fig_spectra = make_subplots(
    rows=1, cols=len(selected_heads),
    subplot_titles=[f"L{l}.H{h} (gap={spectral_data[(l,h)]['spectral_gap']:.2f})" 
                    for l, h in selected_heads],
)

for i, (layer, head) in enumerate(selected_heads, 1):
    sv = spectral_data[(layer, head)]["top_k_sv"]
    fig_spectra.add_trace(
        go.Bar(x=list(range(len(sv))), y=sv, name=f"L{layer}.H{head}",
               marker_color=px.colors.qualitative.Vivid[i % 10]),
        row=1, col=i
    )

fig_spectra.update_layout(
    title="Eigenvalue Spectra: Top-10 Singular Values Per Head",
    template="plotly_dark",
    height=400,
    showlegend=False,
)
fig_spectra.update_xaxes(title_text="Rank")
fig_spectra.update_yaxes(title_text="σ")
fig_spectra.show()

### 4c. Predicted vs. Observed: The Validation
If the Lucier loop truly is power iteration, the empirically observed resonant vector (from EXP_009b) should closely match the mathematically predicted dominant singular vector.

In [ ]:
# ============================================================
# VIS 4c: PREDICTED vs OBSERVED — Cosine Similarity
# ============================================================

if HAS_EMPIRICAL:
    validation_grid = np.zeros((model.cfg.n_layers, model.cfg.n_heads))
    
    for layer in range(model.cfg.n_layers):
        for head in range(model.cfg.n_heads):
            key = f"L{layer}_H{head}"
            if key in empirical_data:
                # Empirical: final vector from the loop
                emp_vec = empirical_data[key]["vectors"][-1]  # last snapshot
                # Predicted: dominant right singular vector
                pred_vec = spectral_data[(layer, head)]["dominant_vector"]
                
                # Cosine similarity (absolute value — direction can flip)
                cos_sim = abs(torch.nn.functional.cosine_similarity(
                    emp_vec.unsqueeze(0).float(),
                    pred_vec.unsqueeze(0).float()
                ).item())
                validation_grid[layer, head] = cos_sim
    
    fig_val = px.imshow(
        validation_grid,
        x=[f"H{h}" for h in range(model.cfg.n_heads)],
        y=[f"L{l}" for l in range(model.cfg.n_layers)],
        color_continuous_scale="Viridis",
        title="Validation: |cos(Empirical, SVD Prediction)| — The Proof",
        labels={"color": "|Cos Sim|"},
        text_auto=".3f",
        aspect="auto",
    )
    fig_val.update_layout(template="plotly_dark", height=600)
    fig_val.show()
    
    # Summary statistics
    flat = validation_grid.flatten()
    print(f"\n{'='*50}")
    print(f"VALIDATION SUMMARY")
    print(f"{'='*50}")
    print(f"Mean |cos sim|:   {flat.mean():.4f}")
    print(f"Median |cos sim|: {np.median(flat):.4f}")
    print(f"Heads > 0.9:      {(flat > 0.9).sum()} / {len(flat)}")
    print(f"Heads > 0.95:     {(flat > 0.95).sum()} / {len(flat)}")
    
    if flat.mean() > 0.9:
        print("\n✓ CONFIRMED: The Lucier loop IS power iteration.")
        print("  The 'resonant drone' is the dominant eigenvector of W_OV.")
        print("  The architecture's voice is mathematically determined.")
    elif flat.mean() > 0.7:
        print("\n~ PARTIAL: Nonlinearities (LayerNorm, attention softmax) introduce deviation.")
        print("  The linear approximation captures the main effect but not all.")
    else:
        print("\n✗ DIVERGENCE: The loop dynamics are dominated by nonlinear effects.")
        print("  The head's behavior cannot be reduced to its OV eigensystem alone.")
else:
    print("Skipping validation (no empirical data). Run EXP_009b first.")
    print("The spectral analysis below is still independently valuable.")

### 4d. The Eigenvoice Token Map
What token does each head's dominant eigenvector predict? Compare to the empirical 'Resonant Voice Map' from EXP_009b.

In [ ]:
# ============================================================
# VIS 4d: EIGENVOICE TOKEN MAP
# ============================================================

md = "# Eigenvoice Map (Predicted from SVD)\n\n"
md += "Each cell shows the top token predicted by the dominant singular vector of W_OV.\n\n"
md += "| | " + " | ".join([f"**H{h}**" for h in range(model.cfg.n_heads)]) + " |\n"
md += "| :--- | " + " | ".join([":---:"] * model.cfg.n_heads) + " |\n"

for layer in range(model.cfg.n_layers):
    row = f"| **L{layer}** |"
    for head in range(model.cfg.n_heads):
        top_token = spectral_data[(layer, head)]["top_tokens"][0][0]
        row += f" `{top_token.strip()}` |"
    md += row + "\n"

display(Markdown(md))

In [ ]:
# ============================================================
# STEP 5: SAVE SPECTRAL ARTIFACTS
# ============================================================

save_dir = os.path.join("..", "_DATA", "EXP_009")
os.makedirs(save_dir, exist_ok=True)

# Save spectral data
save_spectral = {}
for (layer, head), data in spectral_data.items():
    key = f"L{layer}_H{head}"
    save_spectral[key] = {
        "singular_values": data["singular_values"],
        "dominant_vector": data["dominant_vector"],
        "spectral_gap": data["spectral_gap"],
        "top_tokens": data["top_tokens"],
    }

torch.save(save_spectral, os.path.join(save_dir, "009c_spectral_data.pt"))
print(f"[SAVED] {save_dir}/009c_spectral_data.pt")

if HAS_EMPIRICAL:
    torch.save(validation_grid, os.path.join(save_dir, "009c_validation_grid.pt"))
    print(f"[SAVED] {save_dir}/009c_validation_grid.pt")

---
## 5. Theoretical Synthesis

**If validation succeeds (mean cos_sim > 0.9):**

The Lucier experiment has revealed the **Body without Organs** in its most literal form. The weight matrices of each attention head define a fixed geometry — a shape that exists *before* any input is processed. When we run a signal through this geometry repeatedly, all semantic content is stripped away and what remains is **pure structure**: the architectural eigenfunction.

This is **Deterritorialization** made material: the dissolution of content (words, meaning, semantics) until only the substrate (the mathematical structure of the weights) remains.

The **spectral gap** is the measure of how *opinionated* each head is — how quickly it imposes its own voice over the input signal. A large spectral gap is a strongly resonant room. A small gap is an anechoic chamber where the input survives longer.

**The Lucier Isomorphism is now fully grounded:**
> *"The resonant frequencies of a room are analogous to the dominant eigenvectors of the OV circuit. Both are revealed by the same process: iterative re-injection. Both represent intrinsic properties of the medium that are independent of the signal passed through them."*